In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 66
==================================================

Week: 10 of 24
Day: 66 of 168
Date: Wednesday, January 1, 2026
Topic: Deep Q-Networks (DQN) Theory & Architecture

Week 10 Progress:
✅ Day 64: RL Fundamentals - MDP, Value Functions, Bellman Equations (COMPLETED)
✅ Day 65: Q-Learning & Temporal Difference Learning (COMPLETED)
🔄 Day 66: Deep Q-Networks (DQN) Theory & Architecture (TODAY!)
⬜ Day 67: DQN Implementation & CartPole Training
⬜ Day 68: DQN Training & Optimization Techniques
⬜ Day 69: LunarLander Environment Setup & Baseline
⬜ Day 70: LunarLander DQN Training & Week Completion

Progress: 28.6% (2/7 days)

==================================================
🎯 Week 10 Project: Autonomous RL Agent with DQN
- Master Reinforcement Learning fundamentals from scratch
- Implement Deep Q-Networks (DQN) in PyTorch
- Train agent to solve CartPole and LunarLander environments
- Achieve 200+ average reward on LunarLander
- Build foundation for advanced RL algorithms (Week 11)

🎯 Today's Learning Objectives:
1. Understand function approximation in reinforcement learning
   - Why neural networks for Q-values?
   - Q(s,a; θ) - parameterized Q-function with weights θ
   - Gradient descent for Q-Learning updates
   - Challenges: moving targets, correlations, divergence
   - The "deadly triad" of instability

2. Master Deep Q-Networks (DQN) algorithm
   - Experience Replay buffer (break correlations)
   - Target Networks (stabilize learning)
   - DQN loss function and training loop
   - Epsilon-greedy with neural networks
   - Why DQN components are critical

3. Build DQN architecture in PyTorch
   - Neural network design for Q-values
   - Forward pass: state → Q-values for all actions
   - Replay buffer implementation
   - Target network updating mechanism
   - Complete training pipeline

4. Apply DQN to CartPole environment
   - Understand CartPole-v1 (continuous state, discrete actions)
   - Set up Gymnasium environment
   - Train DQN agent from scratch
   - Visualize learning and performance
   - Solve CartPole (195+ average reward over 100 episodes)

📚 Today's Structure:
Part 1 (2h): Function Approximation in RL
Part 2 (2h): DQN Algorithm Deep Dive
Part 3 (3h): PyTorch DQN Implementation
Part 4 (1h): Summary & Next Steps

🎯 SUCCESS CRITERIA:
✅ Explain deadly triad and why tabular Q-Learning fails
✅ Understand Experience Replay mechanism completely
✅ Understand Target Networks and why they stabilize learning
✅ Build complete DQN neural network in PyTorch
✅ Implement replay buffer with sampling
✅ Train DQN on CartPole-v1 successfully
✅ Achieve 195+ average reward (solve CartPole)
✅ Visualize Q-values and learning curves
✅ Ready for advanced DQN techniques tomorrow!

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
!{sys.executable} -m pip install torch torchvision gymnasium numpy matplotlib seaborn pandas tqdm opencv-python -q

print("✅ Libraries installed!")
print("\n" + "=" * 80)

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, namedtuple
import random
from tqdm import tqdm
import time
import os
from copy import deepcopy

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Gymnasium (OpenAI Gym replacement)
import gymnasium as gym

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Matplotlib settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Create results directory
os.makedirs('results/plots', exist_ok=True)
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/videos', exist_ok=True)

print("\n✅ All libraries imported successfully!")
print("=" * 80)
print(f"\n📊 PyTorch version: {torch.__version__}")
print(f"📊 Gymnasium version: {gym.__version__}")
print(f"📊 Random seed set to: 42")
print(f"📁 Results directories created!")
print("=" * 80)

# Test Gymnasium
print("\n" + "=" * 80)
print("🧪 TESTING GYMNASIUM ENVIRONMENT")
print("=" * 80)

env = gym.make('CartPole-v1')
print(f"\n✅ CartPole-v1 environment created!")
print(f"   State space: {env.observation_space}")
print(f"   Action space: {env.action_space}")
print(f"   Max episode steps: {env.spec.max_episode_steps}")
env.close()

print("\n🎯 Ready to build Deep Q-Networks!")
print("=" * 80)

'C:\Program' is not recognized as an internal or external command,
operable program or batch file.


✅ Libraries installed!


📚 IMPORTING LIBRARIES

🖥️  Using device: cpu

✅ All libraries imported successfully!

📊 PyTorch version: 2.9.1+cpu
📊 Gymnasium version: 1.2.2
📊 Random seed set to: 42
📁 Results directories created!

🧪 TESTING GYMNASIUM ENVIRONMENT

✅ CartPole-v1 environment created!
   State space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
   Action space: Discrete(2)
   Max episode steps: 500

🎯 Ready to build Deep Q-Networks!


In [2]:
print("\n" + "=" * 80)
print("🧠 PART 1: FUNCTION APPROXIMATION IN REINFORCEMENT LEARNING")
print("=" * 80)


🧠 PART 1: FUNCTION APPROXIMATION IN REINFORCEMENT LEARNING


In [3]:
# ==================================================
# EXERCISE 1.1: WHY FUNCTION APPROXIMATION?
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: From Tabular to Function Approximation")
print("=" * 80)

"""
📖 THEORY: The Need for Function Approximation

Tabular Q-Learning (Days 64-65):
- Q-table: Store Q(s,a) for each state-action pair
- Works great for small, discrete state spaces (GridWorld)
- Lookup: Q(s,a) directly from table

But what about:
- Atari games: 210×160×3 pixels = 100,800 dimensional state
- Robotics: Continuous joint angles, velocities, positions
- Self-driving: Camera images, LIDAR, GPS, speed, etc.

Problems with Tabular Methods:
1. MEMORY: Cannot store Q-values for all states
   - Atari: 256^100,800 possible states (more than atoms in universe!)
   - Robot: Continuous states (infinite possibilities)
   
2. NO GENERALIZATION: Each state learned independently
   - Similar states don't share knowledge
   - Inefficient learning (must visit every state)
   
3. NEVER SEEN STATES: Cannot handle new states
   - What if agent encounters state never seen before?
   - Tabular: No Q-value available!

Function Approximation Solution:
Instead of storing Q(s,a) for each pair, use a function:

Q(s,a; θ) = neural_network(s, a, weights=θ)

Benefits:
✓ Compact representation (millions of states, thousands of parameters)
✓ Generalization (similar states → similar Q-values)
✓ Works with continuous states
✓ Can handle unseen states (interpolation)
✓ Scales to complex problems (Atari, Go, Robotics)

Why this matters:
- Function approximation enables deep RL
- Neural networks are universal function approximators
- Same algorithm (Q-Learning) but with neural networks!
- This is how AlphaGo, Atari agents, robotics work
"""

print("\n🎯 Tabular vs Function Approximation:")
print("=" * 80)

comparison = """
┌──────────────────────┬─────────────────────┬─────────────────────┐
│      Property        │  Tabular Q-Learning │ Function Approx     │
├──────────────────────┼─────────────────────┼─────────────────────┤
│ Representation       │ Q-table (lookup)    │ Neural network      │
│ Memory               │ O(|S| × |A|)        │ O(parameters)       │
│ State space          │ Small, discrete     │ Large, continuous   │
│ Generalization       │ None                │ Yes                 │
│ Unseen states        │ Cannot handle       │ Can interpolate     │
│ Learning             │ Direct update       │ Gradient descent    │
│ Convergence          │ Guaranteed*         │ Not guaranteed      │
│ Sample efficiency    │ Lower               │ Higher              │
│ Examples             │ GridWorld, Taxi     │ Atari, Robotics     │
└──────────────────────┴─────────────────────┴─────────────────────┘

*Under specific conditions
"""
print(comparison)

print("\n📊 State Space Comparison:")
print("=" * 80)

# Calculate state space sizes
problems = {
    'GridWorld 5×5': {'states': 25, 'type': 'Discrete', 'tabular': 'Yes ✓'},
    'CartPole': {'states': 'Infinite (continuous)', 'type': 'Continuous', 'tabular': 'No ✗'},
    'Atari Pong': {'states': '256^(210×160×3)', 'type': 'Image', 'tabular': 'No ✗'},
    'Robot Arm (7 joints)': {'states': 'Infinite (continuous)', 'type': 'Continuous', 'tabular': 'No ✗'},
    'Go Game': {'states': '10^170', 'type': 'Discrete', 'tabular': 'No ✗'},
}

print(f"{'Problem':<25} {'State Space':<30} {'Type':<15} {'Tabular?':<10}")
print("=" * 80)
for problem, data in problems.items():
    print(f"{problem:<25} {str(data['states']):<30} {data['type']:<15} {data['tabular']:<10}")

print("\n💡 KEY INSIGHT:")
print("=" * 80)
print("""
Function approximation is REQUIRED for:
1. High-dimensional state spaces (images, sensors)
2. Continuous state spaces (robot positions, velocities)
3. Problems where we need generalization

The trade-off:
- Tabular: Guaranteed convergence, but limited to toy problems
- Function Approx: Scales to real world, but learning is harder

Deep Q-Networks (DQN) use neural networks for function approximation!
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: From Tabular to Function Approximation

🎯 Tabular vs Function Approximation:

┌──────────────────────┬─────────────────────┬─────────────────────┐
│      Property        │  Tabular Q-Learning │ Function Approx     │
├──────────────────────┼─────────────────────┼─────────────────────┤
│ Representation       │ Q-table (lookup)    │ Neural network      │
│ Memory               │ O(|S| × |A|)        │ O(parameters)       │
│ State space          │ Small, discrete     │ Large, continuous   │
│ Generalization       │ None                │ Yes                 │
│ Unseen states        │ Cannot handle       │ Can interpolate     │
│ Learning             │ Direct update       │ Gradient descent    │
│ Convergence          │ Guaranteed*         │ Not guaranteed      │
│ Sample efficiency    │ Lower               │ Higher              │
│ Examples             │ GridWorld, Taxi     │ Atari, Robotics     │
└──────────────────────┴─────────────────────┴─────────────────────┘

*Under sp